# 03 — Supervised Learning & Evaluation

**Primary:** Zhilin Zhang  
**Support:** Tuan Wei

Mandatory models: **KNN** and **Decision Tree**.

The RQ requires an explicit comparison between:
1. **property size only**, and
2. **property size + location + amenities**.

## 1. Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

## 2. Load processed data

In [ ]:
candidates = [Path("../data/processed_listings.csv"), Path("data/processed_listings.csv")]
DATA_PATH = next((p for p in candidates if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("Create data/processed_listings.csv first.")

df = pd.read_csv(DATA_PATH)
print("Processed shape:", df.shape)

## 3. Build the final target without using the held-out test set to set the threshold

Contract definition: **high price = price above the training sample's 75th percentile**.

A leakage-safe sequence is:
1. choose the outer train/test row split;
2. calculate Q75 on **training prices only**;
3. label both training and test observations using that fixed training threshold;
4. use stratified CV inside training for hyperparameter tuning.

The rubric strongly prefers a stratified outer split, but the contract's target is not defined until after the training sample exists. Confirm the preferred interpretation with the tutor before final submission. The code below uses a leakage-safe random outer split and prints the resulting class balance so the choice can be explicitly justified.

In [ ]:
PRICE_COL = "price_clean"
ID_COL = "id"

if PRICE_COL not in df.columns:
    raise KeyError(f"{PRICE_COL} must be preserved from preprocessing.")

train_idx, test_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    shuffle=True,
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

price_threshold = train_df[PRICE_COL].quantile(0.75)

train_df["high_price"] = (train_df[PRICE_COL] > price_threshold).astype(int)
test_df["high_price"] = (test_df[PRICE_COL] > price_threshold).astype(int)

print("Training Q75 threshold:", price_threshold)
print("\nTrain class balance:")
display(train_df["high_price"].value_counts().sort_index().to_frame("n").assign(
    pct=lambda x: (x["n"] / x["n"].sum() * 100).round(2)
))
print("\nTest class balance:")
display(test_df["high_price"].value_counts().sort_index().to_frame("n").assign(
    pct=lambda x: (x["n"] / x["n"].sum() * 100).round(2)
))

## 4. Define feature sets

Replace placeholders with the final processed columns agreed by the group. Never include `price_clean` as a predictor of `high_price`.

In [ ]:
SIZE_NUMERIC = [
    # "accommodates", "bedrooms", "bathrooms_num"
]
LOCATION_NUMERIC = [
    # "distance_cbd_km"
]
AMENITY_NUMERIC = [
    # "amenity_count"
]
CATEGORICAL = [
    # Optional agreed categorical variables, e.g. consolidated neighbourhood/property type
]

FEATURE_SETS = {
    "size_only": {
        "numeric": SIZE_NUMERIC,
        "categorical": [],
    },
    "size_location_amenities": {
        "numeric": SIZE_NUMERIC + LOCATION_NUMERIC + AMENITY_NUMERIC,
        "categorical": CATEGORICAL,
    },
}

FEATURE_SETS

## 5. Pipeline builder

All imputing, scaling and one-hot encoding are fitted inside the pipeline to avoid fold leakage.

In [ ]:
def make_preprocessor(numeric_features, categorical_features):
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    transformers = []
    if numeric_features:
        transformers.append(("num", numeric_pipe, numeric_features))
    if categorical_features:
        transformers.append(("cat", categorical_pipe, categorical_features))

    return ColumnTransformer(transformers=transformers)

def make_pipeline(model, numeric_features, categorical_features):
    return Pipeline([
        ("prep", make_preprocessor(numeric_features, categorical_features)),
        ("model", model),
    ])

## 6. Baseline

Use the same held-out test set and the same reported metrics as the trained models.

In [ ]:
TARGET = "high_price"

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(np.zeros((len(train_df), 1)), train_df[TARGET])
baseline_pred = baseline.predict(np.zeros((len(test_df), 1)))

print(classification_report(test_df[TARGET], baseline_pred, digits=4, zero_division=0))
baseline_macro_f1 = f1_score(test_df[TARGET], baseline_pred, average="macro")
print("Baseline macro-F1:", baseline_macro_f1)

## 7. Hyperparameter grids

The rubric requires every tried value's score to be retained, not only the best setting.

In [ ]:
knn_grid = {
    "model__n_neighbors": [3, 5, 7, 11, 15, 21],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2],
}

tree_grid = {
    "model__max_depth": [None, 3, 5, 8, 12],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 5, 10],
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

## 8. Train/tune one model on one feature set

In [ ]:
def tune_and_evaluate(model_name, model, param_grid, feature_set_name):
    spec = FEATURE_SETS[feature_set_name]
    features = spec["numeric"] + spec["categorical"]

    if not features:
        raise ValueError(f"No features configured for {feature_set_name}")

    missing = [c for c in features if c not in train_df.columns]
    if missing:
        raise KeyError(f"Missing features: {missing}")

    X_train = train_df[features]
    y_train = train_df[TARGET]
    X_test = test_df[features]
    y_test = test_df[TARGET]

    pipe = make_pipeline(
        model,
        numeric_features=spec["numeric"],
        categorical_features=spec["categorical"],
    )

    search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        return_train_score=True,
    )
    search.fit(X_train, y_train)

    pred = search.predict(X_test)

    metrics = {
        "model": model_name,
        "feature_set": feature_set_name,
        "best_cv_macro_f1": search.best_score_,
        "test_accuracy": accuracy_score(y_test, pred),
        "test_macro_f1": f1_score(y_test, pred, average="macro"),
        "absolute_macro_f1_improvement_vs_baseline":
            f1_score(y_test, pred, average="macro") - baseline_macro_f1,
        "best_params": search.best_params_,
    }

    print("\n", model_name, feature_set_name)
    print("Best params:", search.best_params_)
    print(classification_report(y_test, pred, digits=4, zero_division=0))
    print("Confusion matrix:\n", confusion_matrix(y_test, pred))

    cv_table = pd.DataFrame(search.cv_results_).sort_values(
        "rank_test_score"
    )

    return {
        "search": search,
        "pred": pred,
        "metrics": metrics,
        "cv_table": cv_table,
        "y_test": y_test.to_numpy(),
    }

## 9. Run all four required RQ comparisons

Uncomment after feature sets are final.

In [ ]:
# experiments = {}
#
# experiments[("KNN", "size_only")] = tune_and_evaluate(
#     "KNN",
#     KNeighborsClassifier(),
#     knn_grid,
#     "size_only",
# )
#
# experiments[("KNN", "size_location_amenities")] = tune_and_evaluate(
#     "KNN",
#     KNeighborsClassifier(),
#     knn_grid,
#     "size_location_amenities",
# )
#
# experiments[("DecisionTree", "size_only")] = tune_and_evaluate(
#     "DecisionTree",
#     DecisionTreeClassifier(random_state=RANDOM_STATE),
#     tree_grid,
#     "size_only",
# )
#
# experiments[("DecisionTree", "size_location_amenities")] = tune_and_evaluate(
#     "DecisionTree",
#     DecisionTreeClassifier(random_state=RANDOM_STATE),
#     tree_grid,
#     "size_location_amenities",
# )

## 10. Uncertainty quantification

Bootstrap one held-out metric for at least one final model. The function below returns a percentile interval for macro-F1.

In [ ]:
def bootstrap_macro_f1_ci(y_true, y_pred, n_boot=2000, alpha=0.05, random_state=42):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    rng = np.random.default_rng(random_state)
    n = len(y_true)
    scores = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        score = f1_score(
            y_true[idx],
            y_pred[idx],
            average="macro",
            zero_division=0,
        )
        scores.append(score)

    scores = np.asarray(scores)
    lower, upper = np.quantile(scores, [alpha/2, 1-alpha/2])

    return {
        "bootstrap_mean": scores.mean(),
        "ci_lower": lower,
        "ci_upper": upper,
        "n_boot": n_boot,
    }

## 11. Results table and evidence checklist

The final table should make the incremental effect of adding location + amenities visible for **both** mandatory models.

Also retain:
- every hyperparameter score tried;
- default vs chosen value for each changed hyperparameter;
- per-class precision/recall/F1;
- baseline and absolute improvement;
- one uncertainty interval;
- feature influence / model behaviour evidence;
- limitations specific to actual results.

In [ ]:
# Example after experiments run:
# summary = pd.DataFrame([v["metrics"] for v in experiments.values()])
# display(summary)
#
# for key, exp in experiments.items():
#     print("\n", key)
#     display(exp["cv_table"][["params", "mean_test_score", "std_test_score", "rank_test_score"]])